# Compare a learned policy with an exact plan

Prerequisites: States, actions, rewards, discounted returns and exploration.

The environment is a five-state chain, starting at 0 and ending at 4. Actions move left or right with clamping at the endpoints. Reaching state 4 gives reward 1; other transitions cost 0.02. Q-learning updates Q(s,a) toward r+γmaxₐ′Q(s′,a′), except that terminal transitions have no bootstrapped continuation.

The companion uses seed 7, 300 episodes, learning rate 0.3, discount 0.9 and ε-greedy exploration at 0.2. It then evaluates the greedy policy without exploration and compares its trajectory with the known shortest plan [0,1,2,3,4]. The undiscounted reward on that path is 0.94; the optimization target uses a discounted return, so those quantities must not be conflated.

This deterministic chain is deliberately easy. A successful seed does not establish robust generalization, safe exploration or a policy for a different transition model. Distinguish termination from a time limit when bootstrapping. Reward design can also encourage shortcuts that conflict with the intended task; a planner baseline helps expose avoidable complexity and wrong objectives.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Experience | (s,a,r,s′,terminal) | Observed transition |
| Update | r+γmax Q(s′,·) | No bootstrap after true termination |
| Reference | Exact shortest plan | Separate learning from planning |

## Predict

The default learned policy should reach the goal along the four-move path. Evaluate multiple seeds separately from training and retain any failed runs.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import systems_models as m

In [ ]:
result=m.reinforcement()
assert result['reached_goal']
assert result['trajectory']==result['optimal_plan']
assert abs(result['undiscounted_return']-.94)<1e-12
print(result)

## Perturb and explain

Introduce a stochastic slip probability and evaluate a frozen policy on several seeds. Compare average return, failure rate and an exact planner that knows the changed transition model.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

At a true terminal transition the Q-learning target is:

1. r only
2. r+γmax Q of an arbitrary next state
3. Always zero

Answer and explanations: 1: There is no continuation after termination. 2: That bootstraps value beyond the defined episode. 3: The terminal reward still matters.

## Primary evidence

- [Sutton and Barto — Reinforcement Learning, second edition](http://incompleteideas.net/book/the-book-2nd.html)
- [Gymnasium — handling time limits](https://gymnasium.farama.org/tutorials/gymnasium_basics/handling_time_limits/)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.